# Анализ ассортимента методом ABC/XYZ

**Контекст и бизнес-задача**

 Для британского интеренет-магазина подарков и товаров для дома нужно провести анализ ассортимента. Руководство считает, что часть ассортимента продается редко, занимает место на складе и усложняет закупки, однако простое удаление "слабых" товаров может привести к потере выручки, ухудшению клиентского опыта или исчезновению важных дополнений к основным покупкам.

 Перед аналитической командой поставлена задача: оценить структуру ассортимента и предложить правила управления товарными позициями. Компания рассматривает сокращение количества SKU примерно на 20%, но ожидает, что аналитик сначала оценит риски такого решения и покажет, какие товары действительно являются кандидатами на сокращение.

 В результате мы должны выяснить какие товары необходимо постоянно держать в ассортименте, какие требуют осторожного управления запасами, а какие можно рассматривать как кандидатов на сокращение или вывод – и какой потенциальный эффект даст такое решение.

**Данные**

Для проекта используется открытый набор данных Online Retail II. Это транзакционные данные интернет-магазина: одна строка соответствует позиции в счете/заказе. В наборе встречаются отмены, отрицательные количества, пропуски и другие особенности реальных данных.

|Поле|Значение|
|:-|:-|
|`Invoice`|Номер счета / заказа.|
|`StockCode`|Код товарной позиции (SKU).|
|`Description`|Описание / название товара.|
|`Quantity`|Количество единиц товара в операции.|
|`InvoiceDate`|Дата и время операции.|
|`Price`|Цена одной единицы товара.|
|`CustomerID`|Идентификатор покупателя.|
|`Country`|Страна клиента.|

## Этап 1. Знакомство с данными и предобработка

### Импорт библиотек

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

### Знакомство с данными

In [ ]:
# Читаем xlsx и склеиваем листы
sheets = pd.read_excel('online_retail_II.xlsx', sheet_name=None, engine='openpyxl')

df = pd.concat(sheets.values(), ignore_index=True)

#### Основная информация о датасете:

In [ ]:
df.info()

In [ ]:
print('Размерность данных:', df.shape)

print('\nПропуски:')
display(
    pd.DataFrame({
        'missing': df.isna().sum(),
        'missing %': (df.isna().mean() * 100).round(2)
    })
)

print(f'''
Диапазон дат:
{df['InvoiceDate'].min()} - {df['InvoiceDate'].max()}
''')

print('Первые пять строк:')
df.head()

- Удалим пробел из названия столбца `Customer ID` для удобства

In [ ]:
df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

- Типы данных в порядке.
- Датасет не так много весит, оптимизация типов данных не требуется.
- В датасете 0.41% пропусков в поле `Description`.
- Огромное количество пропусков (~23%) в столбце `CustomerID`. Можно предположить, что это заказы незарегистрированных пользователей или ошибки в системе. Для ABC/XYZ-анализа они не нужны, можно игнорировать.
- Также стоит отметить, что даты выставляются с точностью до минуты. Это может осложнить выявление дубликатов.

**Диапазон дат:**

Для анализа предлагаю оставить данные за два года (без последнего неполного месяца), чтобы иметь более полную картину и уменьшить влияние сезонности при XYZ-анализе. Однако даже так полностью избавиться от влияния мы не можем.

---
**Проверим, встречаются ли разные описания для одного `StockCode`**

In [ ]:
description_count = df.groupby('StockCode')['Description'].nunique()

print(f'''
В датасете {(description_count.gt(1).sum() 
             / df['StockCode'].nunique()):.2%} StockCode имеют больше одного описания.
''')

- Почти четверть всех `StockCode` имеют больше одного уникального `Description`. Группировать придется только по коду товара.

---
#### Отфильтровываем значения полей, которые напрямую не связаны с продажами и товарами

**Строки с отрицательным `Quantity`**

In [ ]:
print('Количество строк с отрицательным Quantity:', df[df['Quantity']<=0].shape[0],'\n')

df[df['Quantity']<0].head()

- Скорее всего это возвраты и/или другие движения товара, которые не связаны с продажами напрямую. Следовательно, нашему анализу они только вредят и учитывать их не нужно.

**Строки с отрицательным и нулевым `Price`**

In [ ]:
print('Количество строк с отрицательным Price:', df[df['Price']<0].shape[0],'\n')

df[df['Price']<0].head()

- Это списание долгов, для анализа товара не пригодится.

In [ ]:
print('Количество строк с нулевым Price:', df[df['Price']==0].shape[0],'\n')

df[df['Price']==0].sample(5, random_state=42)

- Учитывая то, что строк с нулевой ценой примерно 0.6% от всего датасета и они пересекаются со строками с отрицательным `Quantity`, их удаление не сильно повлияет на XYZ-анализ.

**Нетоварный `StockCode`**

Попробуем найти нетоварные `StockCode`. Их можно было бы исследовать отдельно, но для целей нашего анализа они не нужны.

In [ ]:
import string
letters = tuple(string.ascii_letters) # Создает кортеж из букв английского алфавита

mask = df['StockCode'].astype(str).str.startswith(letters)

df[mask]['StockCode'].unique()

In [ ]:
# Список нетоварных кодов
non_product_codes = [
    'POST', 'DOT', 'M', 'm', 'C2', 'BANK CHARGES', 'TEST001', 'TEST002', 'PADS', 'ADJUST', 'ADJUST2', 'S', 'B'
]

Скорее всего нашел не все, но даже так результат анализа будет немного чище.

Некоторые коды, которые не попали в список (например, разного рода гифты и комиссии) и так отфильтруются, потому что имеют либо нулевую цену, либо отрицательное количество.

Да и те, что попали в список, имеют пересечение со множеством отрицательных `Quantity` и нулевых `Price`.

---
#### Дубликаты

**Для начала проверим дубликаты по всем полям**

In [ ]:
print(f'''
Всего дубликатов по всем столбцам: {df.duplicated().sum()}.
Что составляет {df.duplicated().sum()/len(df):.2%} от всех строк в датасете.
''')

In [ ]:
duplicates = df[df.duplicated(keep=False)]
print("Несколько строк с дубликатами:\n") 
display(duplicates.head(15))

- Видно, что в одном заказе могут повторяться позиции с одинаковым кодом товара, количеством и ценой. Я точно не уверен, можно ли это считать техническими ошибками или это особенность работы интернет-магазина.
- Нужно проверить дубликаты по всем столбцам, кроме `Quantity`, чтобы узнать, могут ли задваиваться позиции c разным количеством в заказе. Перед этим создадим копию без полных дубликатов, чтобы они не мешались.

In [ ]:
# Создадим копию датафрейма без полных дубликатов
df_clean = df.drop_duplicates().copy()

df_clean[
    df_clean.duplicated(
        subset = df_clean.drop('Quantity', axis=1).columns,
        keep=False
        )
].head(15)

- Теперь очевидно, что в заказе может встречаться одинаковый товар с разным количеством. Удаление таких строк может сильно исказить результат нашего анализа.
- Следовательно, если мы не удаляем строки с разным количеством товара, то и удалять строки с одинаковым количеством у нас нет резона, потому что это могут быть не технические ошибки.

**Вывод:**

- Для адекватного решения об удалении полных дубликатов нужно разбираться в движке конкретно этого интернет-магазина. У нас такой информации нет.   

---
**Проверим дубликаты по другим сабсетам в очищеном датасете**

*Дубликаты с разными ценами*

In [ ]:
# Проверим количество дубликатов всем полям, кроме Price
print('Количество дубликатов всем полям, кроме Price:')
df_clean.duplicated(
    subset = df_clean.drop('Price', axis=1).columns
).sum()

Посмотрим на несколько строк с этими дубликатами

In [ ]:
df_clean[
    df_clean.duplicated(
        subset = df_clean.drop('Price', axis=1).columns,
        keep=False
        )
].head(20)

- Здесь мы так же наблюдаем строки с одинаковым товарным кодом в одном заказе, но цена при этом разная. Возможно так обозначаются скидки при определенном количестве одинакового товара в заказе. В любом случае для ABC-анализа нужно агрегировать данные по коду, так что эти повторяющиеся строки не помешают.

*Дубликаты с разным описанием*

In [ ]:
# Посмотрим на дубликаты с разным описанием
df_clean[
    df_clean.duplicated(
        subset = df_clean.drop('Description', axis=1).columns,
        keep=False
        )
]

- Это странно, но не критично. Группировать все равно будем только по коду товара.

---
#### Пропуски в поле `Description`

**Рассмотрим несколько строк с пропусками**

In [ ]:
df[df['Description'].isna()].head(10)

- Сразу бросается в глаза нулевая цена и отсутствующие `CustomerID`.
- Нужно проверить справедливо ли это для всех строк с пропусками в `Description`.

In [ ]:
print('Суммарная цена и количество уникальных значений CustomerID в строках с пропуском в поле Description:')
df[df['Description'].isna()].agg({
    'Price': 'sum',
    'CustomerID': 'nunique'
})

- У всех строк с пропусками в описании нулевая цена и отсутствует код клиента. Это не может быть просто совпадением, и при этом назначение таких строк нам неизвестно.
- Они все равно не будут учитываться в анализе из-за нулевой цены.

---
### Итог предобработки

**Для дальнейшего анализа предлагаю:**
- Оставить диапазон дат с 2009-12-01 по 2011-12-01.
- Удалить строки с отрицательным количеством, потому что они не относятся к продажам. Их нужно анализировать отдельно.
- Оставить только строки с ценой выше нуля по той же причине.
- Строки с отсутствующим описанием не будут учитываться, потому что имеют нулевой `Price`.
- Убрать строки с нетоварными `StockCode`.

In [ ]:
sales = df[
      (df['InvoiceDate'] >= '2009-12-01')
    & (df['InvoiceDate'] < '2011-12-01')
    & (df['Quantity'] > 0)
    & (df['Price'] > 0)
    & (~df['StockCode'].isin(non_product_codes))
].copy()

print(f'После очистки осталось {sales.shape[0]} строк\n')
sales.sample(5, random_state = 42)

---
## Этап 2. Базовая картина бизнеса

### Рассмотрим масштаб продаж магазина в выбранный период.

In [ ]:
# Считаем выручку
sales['Revenue'] = sales['Quantity'] * sales['Price']

# Добавим индикаторы месяца и года
sales['Month'] = sales['InvoiceDate'].dt.to_period('M')
sales['Year'] = sales['InvoiceDate'].dt.year

sales.sample(5, random_state = 42)

### Сделаем таблицу с основной информацией.

In [ ]:
# Сделаем серию с основными показателями
summary = pd.Series({
    'Уникальные SKU': sales['StockCode'].nunique(),
    'Уникальные заказы': sales['Invoice'].nunique(),
    'Уникальные CustomerID': sales['CustomerID'].nunique(),
    'Выручка': sales['Revenue'].sum(),
    'Количество единиц': sales['Quantity'].sum(),
    'Средний чек': sales.groupby('Invoice')['Revenue'].sum().mean(),
}).to_frame('summary')

summary

### Рассмотрим динамику выручки по месяцам.

In [ ]:
monthly_revenue = sales.groupby('Month', as_index = False)['Revenue'].sum()
monthly_revenue['Month'] = monthly_revenue['Month'].astype('str')

plt.figure(figsize=(11, 4))
sns.lineplot(
    data = monthly_revenue,
    x = 'Month',
    y = 'Revenue',
    marker = 'o'
)

plt.title('Динамика выручки по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Выручка')

plt.grid(alpha = 0.4)
plt.xticks(rotation = 45)
plt.tight_layout()

- Наблюдается повышение выручки в последнем квартале каждого года.

---
## Этап 3. ABC-анализ

### Кривая Парето

In [ ]:
abc = sales.groupby(
    'StockCode',
    as_index = False
).agg(
    Revenue = ('Revenue', 'sum'),
    Quantity = ('Quantity', 'sum'),
    Orders = ('Invoice', 'nunique')
).sort_values(by = 'Revenue', ascending = False).reset_index(drop = True)


abc['RevenueShare'] = abc['Revenue'] / abc['Revenue'].sum()
abc['CumRevenueShare'] = abc['RevenueShare'].cumsum()
abc['CumSKUShare'] = (
    np.arange(1, abc.shape[0]+1)
    / abc.shape[0]
)

sku_for_80 = abc[abc["CumRevenueShare"] <= 0.8].index[-1] + 1

plt.figure(figsize = (9, 4))
sns.lineplot(
    x = abc['CumSKUShare'] * 100,
    y = abc['CumRevenueShare'] * 100,
)

plt.axhline(80, linestyle = '--', alpha = .4)
plt.grid(axis = 'x', alpha = .4)

plt.title('Кривая Парето по товарам')
plt.xlabel('Накопительная доля SKU, %')
plt.ylabel('Накопительная доля выручки, %')
plt.tight_layout()
plt.show()

print(f'''
Для формирования примерно 80% выручки нужно {sku_for_80} SKU из {abc.shape[0]}.
''')

- Принцип Парето практически идеально соблюден. В группе `A` будет 1040 SKU.

### Распределяем SKU по группам.

ABC-анализ в этом проекте используется для оценки экономической значимости товара. Базовая метрика – выручка за выбранный период.

Для границ определения групп используем классическое распределение:
 1. `A` - товары, которые формируют примерно первые 80% выручки;
 2. `B` - следующие 15%;
 3. `С` - оставшиеся.

In [ ]:
abc['ABC'] = np.select(
    [
        abc['CumRevenueShare'] <= 0.8,
        abc['CumRevenueShare'] <= 0.95
    ],
    ['A', 'B'],
    default = 'C'
)

abc.head()


### Доля каждой группы в общем количестве SKU

In [ ]:
abc_summary = abc.groupby(
    'ABC',
    as_index = False
).agg(SKU_count = ('StockCode', 'nunique'))


abc_summary['SKU_share'] = (
    abc_summary['SKU_count']
    / abc_summary['SKU_count'].sum()
)

abc_summary.style.format(
    {'SKU_share': '{:.2%}'}
)

- Больше половины ассортимента приносят лишь 5% дохода.

---
## Этап 4. XYZ-анализ
XYZ показывает стабильность спроса. Используем количество проданных единиц по месяцам и коэффициент вариации:

$$CV = \frac{\text{std}}{\text{mean}}$$

Ключевой момент: если товар не продавался в месяце, это должно быть `0`, а не отсутствие строки.

In [ ]:
monthly_quantity = sales.groupby(
    ['StockCode', 'Month'],
    as_index = False
)['Quantity'].sum()

xyz_pivot = monthly_quantity.pivot_table(
    index = 'StockCode',
    columns = 'Month',
    values = 'Quantity',
    aggfunc = 'sum'
).reindex(
    columns = sales['Month'].unique(),
    fill_value = 0
).fillna(0)

xyz_pivot['mean_qty'] = xyz_pivot.mean(axis = 1)
xyz_pivot['std_qty'] = xyz_pivot.std(axis = 1, ddof = 0)
xyz_pivot['CV'] = xyz_pivot['std_qty'] / xyz_pivot['mean_qty']

xyz_pivot['XYZ'] = np.select(
    [
        xyz_pivot['CV'] <= 0.5,
        xyz_pivot['CV'] <= 1
    ],
    ['X', 'Y'],
    default = 'Z'
)

xyz_pivot.columns.name = None # Чтобы над столбцом с индексами не было надписи Month 
xyz = xyz_pivot[['std_qty', 'mean_qty', 'CV', 'XYZ']].reset_index()

xyz_summary = xyz.groupby(
    'XYZ',
    as_index = False
).agg(SKU_count = ('StockCode', 'nunique'))


xyz_summary['SKU_share'] = (
    xyz_summary['SKU_count']
    / xyz_summary['SKU_count'].sum()
)

xyz_summary.style.format(
    {'SKU_share': '{:.2%}'}
)

- В нашем датасете `~80%` товаров имеют сезонную или непостоянную природу, что неудивительно, учитывая график динамики выручки по месяцам.

---
## Этап 5. Объединенная матрица ABC/XYZ
### Строим матрицу пересечений групп ABC и XYZ.

In [ ]:
abc_xyz = abc.merge(
    xyz[['StockCode', 'CV', 'XYZ']],
    on = 'StockCode'
)
abc_xyz['ABC_XYZ'] = abc_xyz['ABC'] + abc_xyz['XYZ']

abc_xyz.head()

In [ ]:
matrix = pd.crosstab(
    abc_xyz['ABC'],
    abc_xyz['XYZ']
)

matrix

- Интересно, что в каждой группе из `ABC` у наибольшей части товаров высокая изменчивость.
- При этом в группах с низким и средним коэффициентом вариации (`X` и `Y`) преобладают товары с высокими показателями выручки.
- А в группе с высокой вариативностью, наоборот, преобладают товары с низкой выручкой. Что логично, ведь если они продаются редко то у них и меньше шансов набрать выручку.

По этой матрице видно, что `~80%` товаров в датасете имеют сезонную природу, `~60%` из которых ещё и с низкой выручкой, но это не повод от них избавляться. Нужно посмотреть на каждую группу детальнее.

### Базовые метрики для каждой группы

In [ ]:
abc_xyz_summary = abc_xyz.groupby(
    'ABC_XYZ',
    as_index = False
).agg(
    SKU = ('StockCode', 'nunique'),
    TotalRevenue = ('Revenue', 'sum'),
    TotalQuantity = ('Quantity', 'sum'),
    TotalOrders = ('Orders', 'sum'),
    RevenueShare = ('RevenueShare', 'sum')
)

abc_xyz_summary['SKUShare'] = abc_xyz_summary['SKU'] / abc_xyz_summary['SKU'].sum()

In [ ]:
sales_abc_xyz = pd.merge(
    sales[['StockCode', 'Price', 'CustomerID' , 'Month']],
    abc_xyz[['StockCode', 'ABC_XYZ']],
    on = 'StockCode',
    how = 'left'
)

abc_xyz_indicators = sales_abc_xyz.groupby(
    'ABC_XYZ',
    as_index = False
).agg(
    Customers = ('CustomerID', 'nunique'),
    ActiveMonths = ('Month', 'nunique'),
    AvgPrice = ('Price', 'mean')
)

abc_xyz_summary = abc_xyz_summary.merge(
    abc_xyz_indicators,
    on = 'ABC_XYZ'
)

In [ ]:
abc_xyz_summary

- Все получившиеся группы были активны на протяжении всех месяцев присутствующих в датасете.
- Судить только по доле выручки неправильно, потому что эта доля так же зависит от количества товаров пропавших в группу. Лучше интерпретировать долю выручки относительно доли общем в количестве товаров.

---
## Этап 6. Интерпретация и стратегия управления

|Группа|Cтратегия управления|
|:-|:-|
|`AX`|Высокая выручка, стабильный спрос и товаров не так много: держать в наличии.|
|`AY`|Высокая выручка при колеблющемся спросе. Эта группа - лидер почти по всем показателям, держать в наличии с учетом сезонности товаров.|
|`AZ`|Большая выручка и большая вариативность. Закупка строго с учетом сезонности товара.|
|`BX`|Средняя выручка и стабильный спрос. В этой группе всего 21 товар. Желательно держать в наличии.|
|`BY`|Средняя выручка с колебаниями в спросе. Осторожное управление с учетом сезонного спроса.|
|`BZ`|Средняя выручка и большие колебания в спросе. В этой группе 1001 товар. Нужен серьезный контроль запаса с учетом объемов и сезонности. Кандидат на дополнительное исследование.|
|`CX`|Низкая выручка, но стабльные продажи. В этой группе всего 1 товар - приглашения на вечеринку. Желательно держать в наличии, если от этого нет никаких издержек.|
|`CY`|Низкая выручка и умеренная вариативность. Нужно дополнительное исследование метрик.|
|`CZ`|Низкая выручка и высокая вариативность. Это самая многочисленная группа, в ней лучше всего искать кандидатов на вывод из ассортимента.|

## Этап 7. Cписок кандидатов на сокращение

### Рассмотрим графики доли SKU и доли выручки по группам ABC/XYZ

In [ ]:
# Получаем уникальные группы
unique_groups = abc_xyz_summary['ABC_XYZ'].unique()
# Получаем палитру размером с кол-во групп
colors = sns.color_palette('tab10', n_colors=len(unique_groups))
# Закрепляем цвета за группами
group_palette = dict(zip(unique_groups, colors))

share_title = [
    ('SKUShare', 'Доля SKU (%)'),
    ('RevenueShare', 'Доля выручки (%)')
]

# Построим два графика
for (share, title) in share_title:
    df_plot = abc_xyz_summary.sort_values(by = share, ascending = False)

    plt.figure(figsize=(9,4))
    sns.barplot(
        x = df_plot['ABC_XYZ'],
        y = df_plot[share] * 100,
        palette = group_palette
    )

    plt.grid(alpha=0.4, axis='y')
    plt.ylim(0, 50)
    
    plt.xlabel('Группа ABC/XYZ')
    plt.ylabel(title)
    plt.title(f'{title} по группам ABC/XYZ')

    plt.tight_layout()

- Поиск кандидатов на удаление стоит проводить внутри группы `CZ`. К ней принадлежит почти `50%` из всех товаров и при этом её доля в выручке меньше `5%`.

### Поиск кандидатов на вывод

 1. Важно, чтобы эти товары не были новыми. Нужно взять те из них, которые продавались в течении первого года из двух, при этом, чтобы этот товар попал под первый пик на графике динамики выручки. Т.е. товары нужно искать по фильтру `2009-12 - 2011-01`.
 2. Дальше нужно выбрать из них тех, кто показал меньше всего выручки.
 3. Далее желательно посмотреть на рост год к году и выявить тех, у кого он наименьший. Тут стоит отметить, что второй год у нас неполный, поэтому мы смотрим не на сами числа, а сравниваем их между собой.
 4. При этом `SKU` должно быть примерно `20%` от общего пула товаров. Это и будут наши кандидаты на вылет.

In [ ]:
abc_xyz

In [ ]:
# Добавим в sales метку группы
sales_merge = sales.merge(
    abc_xyz[['StockCode','ABC', 'XYZ', 'ABC_XYZ']],
    on = 'StockCode',
    how = 'left'
)
# Составим список товаров для дальнейшей проверки и отфильтруем датафрейм
CZ = sales_merge[
      (sales_merge['ABC_XYZ']=='CZ')
    & (sales_merge['InvoiceDate'] >= '2009-12-01')
    & (sales_merge['InvoiceDate'] < '2011-01-01')
]['StockCode'].unique()

sales_filtered = sales_merge[sales_merge['StockCode'].isin(CZ)]

In [ ]:
# Считаем общую выручку
total_revenue = sales_filtered.groupby(
    'StockCode',
    as_index=False
).agg(TotalRevenue = ('Revenue', 'sum'))

# Выручка по годам
yearly_revenue = sales_filtered[
    sales_filtered['Year']!=2009
].pivot_table(
    index = 'StockCode',
    columns = 'Year',
    values = 'Revenue',
    aggfunc = 'sum',
    fill_value = 0
).reset_index().rename(
    columns={
        2010: 'Revenue2010',
        2011: 'Revenue2011'
    }
)

yearly_revenue.columns.name = None
yearly_revenue.head(5)

- Некоторые `StockCode` перестали продаваться в 2011. Возможно их просто перестали заказывать на склад, возможно перестали покупать клиенты. У нас нет информации по складским остаткам.
- Нужно узнать количество таких позиций.

In [ ]:
print(
    'Количество SKU, которые перестали продаваться в 2011:',
    yearly_revenue[yearly_revenue['Revenue2011']==0]['StockCode'].nunique()
)

- Это первые кандидаты на вывод.
- Дальше проверим общую выручку и рост год к году

In [ ]:
# Считаем рост выручки год к году без учета нулевой выручки за 2011
yearly_revenue_filtered = yearly_revenue[yearly_revenue['Revenue2011']!=0].copy()

yearly_revenue_filtered['Growth'] = np.where(
    yearly_revenue_filtered['Revenue2010'] > 0,
    yearly_revenue_filtered['Revenue2011']
    / yearly_revenue_filtered['Revenue2010'] - 1,
    np.nan
)

# Добавим к датафрейму информацию о полной выручке
for_deletion_df = yearly_revenue_filtered.merge(
    total_revenue,
    on = 'StockCode',
    how = 'inner'
)

for_deletion_df.head()

**Финальный список**

Предлагаю отфильтровать `SKU` по двум критериям:
1. Общая выручка меньше медианы среди отобраных товаров.
2. Отрицательный рост 2011 к 2010 гг.

In [ ]:
products_for_deletion = for_deletion_df[
    (for_deletion_df['TotalRevenue'] < for_deletion_df['TotalRevenue'].quantile(0.5))
    & (for_deletion_df['Growth'] < 0)
]['StockCode'].unique()

# Так же добавим в список SKU  с нулевой выручкой в 2011
products_for_deletion = np.append(
    products_for_deletion,
    yearly_revenue[yearly_revenue['Revenue2011']==0]['StockCode'].unique()
)

# Финальная таблица
df_filtered = sales_merge[sales_merge['StockCode'].isin(products_for_deletion)]

df_for_deletion = df_filtered.groupby(
    ['StockCode', 'Description', 'ABC', 'XYZ', 'ABC_XYZ'],
    as_index = False
).agg(
    TotalRevenue = ('Revenue', 'sum'),
    TotalQuantity = ('Quantity', 'sum'),
    TotalOrders = ('Invoice', 'nunique'),
    TotalCustomers = ('CustomerID', 'nunique'),
    ActiveMonths = ('Month', 'nunique')
)

df_for_deletion['RevenueShare'] = df_for_deletion['TotalRevenue'] / sales['Revenue'].sum()

df_for_deletion.head()

## Этап 8. Потенциальный эффект

In [ ]:
print(f'''
Изначально было взято в работу {sales['StockCode'].nunique()} SKU.

Предлагаю к удалению {len(products_for_deletion)} SKU.

Доля списка кандидатов для вывода от общего количества SKU: {len(products_for_deletion) / sales['StockCode'].nunique():.2%}.

Доля исторической выручки этих товаров: {df_for_deletion['TotalRevenue'].sum() / sales['Revenue'].sum():.1%}.
''')

### Финальный вывод:

Предлагается проверить на вывод `23.9% SKU`. В них входят товары, которые перестали продаваться в 2011 году, имеют отрицательный рост 2011 к 2010 гг и суммарную выручку меньше медианы из группы `CZ` (без учета тех, кто в 2011г имел нулевую выручку).

На них приходится `1,5%` исторической выручки. Из списка исключены новые товары.

### Ограничения в исследовании

В нашем распоряжении был только датасет с продажами, однако, для адекватного принятия решения по выводу товара этого мало.

В реальном управлении ассортиментом важны себестоимость и маржа, остатки на складе, стоимость хранения, срок поставки, минимальная партия поставщика, возвраты, срок годности и т.п.

Следовательно наши данные о продажах не равны истинному спросу. Если товара не было на складе, в транзакциях тоже будет ноль продаж. Поэтому низкие продажи без данных о наличии нельзя автоматически трактовать как отсутствие интереса покупателей.